# Introduction

`keecas` is a module for performing symbolic and units-aware calculations. It has been developed to be used mainly in a jupyter notebook's interactive environments, which then could be rendered as pdf by [quarto](https://quarto.org/).

`keecas` leverage some well known python modules, wrapping some of their functions with sensisble defaults, aiming to reduce boiler plate code and be quick to use. Some of the modules used are:

`sympy`
: for all symbolic expression and computation

`pint`
: convenient unit registry; `pint` quantities can be converted in `sympy.physics.units` directly

`pipe`
: some of the most used `sympy` functions are wrapped in `pipe` object, so that they can be used in a sequential fashion



## Quick Start

Let's start with a bare minimum example: let's calculate the maximum bending moment ($M_{Sd}$) for a beam of span $l$, simply supported at the end (pinned), uniformly loaded ($q$).

In [1]:
# import everything from keecas
from keecas import *
from keecas import pc, show_eqn, symbols, u

u.formatter.default_format = ".2f~P"
# some notable imports:
# - `symbols`: function to define symbols from sympy module
# - `u`: unit registry from pint module
# - `pc`: short for `keecas.pipe_command` where are the most used sympy function are defined as pipe object

In [2]:
config.col_wrap = [None]
config.display.print_label = True

In [3]:
# define the symbols with symbols() from `sympy`
q, l, M_Sd = symbols(r'q, l, M_{Sd}')

# define the parameters with units support provided by `pint` (`u` is the unit registry)
_p = {
    q: (5*u('kN/m')),
    l: S(400*u.cm),
}

# define the symbolic expression
_e = {
    M_Sd: "q*l^2/8" # the expression is written as a simple string
    | pc.parse_expr, # function that will parse the expression (from `sympy`)
}

# evaluate the expression in `_e`
_v = {
    k: v
    | pc.subs(_e|_p) # subistitute the symbol contained in the joined dict _e|_p (from `sympy`)
    | pc.convert_to([u.kN, u.m]) # convert the expression (from `sympy`) to the preferred units (list of `pint` units)
    | pc.N # elavuate the expression as decimal (from `sympy`)
    for k,v in _e.items() # get all the expression contained in _e
}

# add some description
_d = {
    M_Sd: "max bending moment",
    q: "uniform load",
    l: "span of the beam",
}

# create the latex expression (amsmath) to be displayed as Markdown object
show_eqn(
    [
        _p|_e,
        _v,
        _d,
        # _d,
        # _d,
    ], # list of dict to be shown
    float_format='{:.2f}', # precision of floats
    debug=True,
    # cell_formatter=FormatterChain([format_pint, format_sympy]),
)

\begin{align}
q & = \dfrac{5.00{\,}\text{kN}}{\text{m}} &   & \quad\text{uniform load}  \\[8pt]
 l & = 400{\,}\text{cm} &   & \quad\text{span of the beam}  \\[8pt]
 M_{Sd} & = \dfrac{q{\,}l^{2}}{8} & = 10.00{\,}\text{kN}{\,}\text{m} & \quad\text{max bending moment} 
\end{align}


<IPython.core.display.Latex object>

We can have a look at the latex code by passing `debug=True` to the function or setting globally `config.display.debug=True`

In [4]:
from keecas.dataframe import Dataframe, create_dataframe

create_dataframe(
    Dataframe({q:[None, '2.f', '.3f']}),
    width=4,
    keys=_p.keys(),
    # default_value=None
)

Dataframe({q: [None, '2.f', '.3f', None], l: [None, None, None, None]}, shape=(2, 4))

In [5]:
# options.DEBUG = True

show_eqn(
    [_p|_e, _v, _d], # list of dict to be shown
    float_format=['{:.2f}', '.3f'], # precision of floats
    # col_wrap=[ # columns wrappers
    #     None, # wrapping for the 1st column (key of first dict in the main argument list -> `_p|_e` )
    #     '=', # add '=' sign before the 2nd column (value of first dict in the main argument list -> `_p|_e`)
    #     '=', # add '=' sign before the 3rd column (value of second dict in the main argument list -> `_v`)
    #     # (r'\quad(', ')') # wrap the 4th column with '\quad(' on the left and ')' on the right (value of the third dict in the main argument list -> `_d`)
    # ],
    # col_wrap={E: ['0', '1']},
    debug=True,
    environment='align',
)

\begin{align}
q & = \dfrac{5.000{\,}\text{kN}}{\text{m}} &   & \quad\text{uniform load}  \\[8pt]
 l & = 400{\,}\text{cm} &   & \quad\text{span of the beam}  \\[8pt]
 M_{Sd} & = \dfrac{q{\,}l^{2}}{8} & = 10.000{\,}\text{kN}{\,}\text{m} & \quad\text{max bending moment} 
\end{align}


<IPython.core.display.Latex object>

## A more structured example: Simple Beam

In the previous example all the `dict` were prepended by `_`: that because those dict are not really meant to be preserved further than the cell they are defined. They are meant to only collect the expressions/parameters of the cell to be displayed, and then discarded. To preserve all the expression/parameters of the notebook wwe will initialize a named dict.


In [6]:
# this dict will act as namespace for all the expressions/parameters defined in this file (could be used in other notebook)
simple_beam = {
    'parameters' : (params := {}), # params is the dict that will collect all the params of this notebook
    'expressions': (eqn := {}), # eqn is the dict that will collect all the eqn of this notebook
}

We will calculate the notable values for a simple supported beam with a uniform load in ULS condition for bending and shear, and SLS condition for deflection.

The beam is characterized by this parameters:

In [7]:
l, b_i = symbols(r"l, b_{i}")

_p = {
    l: 6*u.m,
    b_i: 3*u.m,
}
params.update(_p) # save the parameters in the notebook dict

_d = {
    l: "span of the beam",
    b_i: "width of influence",
}

show_eqn([_p, _d], col_wrap=[None, None, "&"], environment="cases")

<IPython.core.display.Latex object>

### Actions

Let's calculate the load applied to a simple beam.

#### Loads

In [8]:
G_1, G_2, Q_k = symbols(r"G_{1}, G_{2}, Q_{k}")


# define the loads in whatever units you want
_p = {
    G_1: 2.5 * u.kPa,
    G_2: 300 * u("daN/m^2"),
    Q_k: 4 * u.kN / u.m**2,
}
params.update(_p)  # save the parameters in the notebook dict

_d = {
    G_1: "permanent loads",
    G_2: "permanent non structural",
    Q_k: "live loads",
}

# NOTE: values will be rendered as `pint` quantities because they are not being converted yet into `sympy` objects
show_eqn([_p, _d], col_wrap=[None, None, "&"])

<IPython.core.display.Latex object>

#### Safety coefficients for loads

In [9]:
gamma_G1, gamma_G2, gamma_Qk = symbols(r'\gamma_{G_{1}}, \gamma_{G_{2}} , \gamma_{Q_{k}}')

_p = {
    gamma_G1: 1.3,
    gamma_G2: 1.5,
    gamma_Qk: 1.5,
}
params.update(_p)

# since _p is being redefined in this cell, it will only display the content of this cell
show_eqn(_p)

<IPython.core.display.Latex object>

#### Applied forces

The applied load are multiplied for the width of influence $b_i$ of the beam:

In [10]:
F_d, F_k = symbols(r"F_{d}, F_{k}")

_e = {
    F_k: "(G_1 + G_2 + Q_k)*b_i" | pc.parse_expr,
    F_d: "(gamma_G1*G_1+gamma_G2*G_2+gamma_Qk*Q_k)*b_i" | pc.parse_expr,
}
eqn.update(_e)  # save the expressions in the notebook dict

_v = {
    k: (
        v
        | pc.subs(eqn | params)
        | pc.convert_to([u.kN, u.m])
        | pc.N
        | pc.as_two_terms(as_mul=True) # this will display nicely the values and units in the show_eqn function
    )
    for k, v in _e.items()
}

_d ={
    F_k: "(SLS)",
    F_d: "(ULS)",
}

show_eqn([_e, _v, _d], float_format='{:.2f}')

<IPython.core.display.Latex object>

## ULS: Bending Moment and Shear

We calculate the bending moment and shear in ULS condition:



In [11]:
M_Sd, V_Sd = symbols(r"M_{Sd}, V_{Sd}")

_e = {
    M_Sd: "F_d * l^2 / 8" | pc.parse_expr,
    V_Sd: "F_d * l / 2" | pc.parse_expr,
}
eqn.update(_e)  # save the expressions in the notebook dict

_v = {
    k: (
        v
        | pc.subs(eqn | params)
        | pc.convert_to([u.kN, u.m])
        | pc.N
        | pc.as_two_terms(as_mul=True) # this will display nicely the values and units in the show_eqn function
    )
    for k, v in _e.items()
}

# we can be finer with the float format options
_f = {
    M_Sd: '{:.3f}',
    V_Sd: '{:.0f}',
}

show_eqn(
    [_e, _v, _d],
    float_format=_f,
    )

<IPython.core.display.Latex object>

## SLS: deflection

We calculate the deflection of the beam with the following equation:

In [12]:
f, E, J = symbols(r"f , E, J")

# since we want to collect display the expression as two separate terms (5/384) and (F_k * l^4 / (E *J)), we will wrap in `S()` (singleton from sympy); uncomment the line to see the different display results
_e = {
    # f : " (5/384) * (F_k * l^4 / (E *J))" | pc.parse_expr,
    f : " S(5/384) * S(F_k * l^4 / (E *J))" | pc.parse_expr,
    # f : " N(5/384) * S(F_k * l^4 / (E *J))" | pc.parse_expr, # th N() will evaluate the fraction to a decimal

}
eqn.update(_e)

show_eqn(_e)

<IPython.core.display.Latex object>

Assuming a `IPE270` beam, the deflection results

In [13]:
_p = {
    E: 210000 * u("MPa"),
    J: 8356*u.cm**4,
}
params.update(_p)

_v = {
    k: (
        v
        | pc.subs(eqn | params)
        | pc.convert_to([u.kN, u.mm])
        | pc.N
    )
    for k, v in _e.items()
}

# I evaluate a simple expression that has not its own symbol defined
__v = {
    k: k | pc.subs(eqn | params) | pc.convert_to([u.kN, u.mm]) | pc.N for k in [l/f]
}

_d = {
    l/f: "ratio span of the beam / deflection",
}

show_eqn([_p|_v|__v, _d], float_format='{:.2f}')

<IPython.core.display.Latex object>

In [ ]:
config.language.language = "it"
show_eqn(
    {
        'verifica' : check(1.1, 1.0, template='boxed', language='it'),
    },
    debug=True,
    environment='cases',
)

In [15]:
show_eqn(
    {
        'piecewise': '''Piecewise(
            (1, x<0), 
            (2, True),
        )
        ''' | pc.parse_expr,
    },
)

<IPython.core.display.Latex object>

# Labels

Various showcase of labeling generation:

In [16]:
# setup
from keecas import generate_label, generate_unique_label

config.display.katex = True

J, E, a, b, c_0 = symbols(r"J, E, a, b, c_0")

# dictionary to be displayed by show_eqn
_p = {
    J: 123*u.cm**4,
    E: 456*u.MPa,
    a: "b+c_0/2" | pc.parse_expr,
}

## Manual assignment

In [17]:
# when manually assigning the labels, they should be LaTeX safe
_l = {
    J: "eq-moment-of-inertia",
    E: "eq-modulus-of-elasticity",
    a: "eq-an-expression",
}

# pass _l to show_eqn
show_eqn(_p, label=_l)


J: eq-moment-of-inertia
E: eq-modulus-of-elasticity
a: eq-an-expression


<IPython.core.display.Latex object>

## Partial automatic generation

### Non unique labels

In [18]:
# when manually assigning the labels, they should be LaTeX safe
_l = {
    J: "moment-of-inertia",
    E: "modulus-of-elasticity",
    a: "an-expression",
}


# generate the labels from the description
_l = generate_label(_l)
print(_l)

# pass _l to show_eqn
show_eqn(_p, label=_l)

{J: 'eq-moment-of-inertia', E: 'eq-modulus-of-elasticity', a: 'eq-an-expression'}
J: eq-moment-of-inertia
E: eq-modulus-of-elasticity
a: eq-an-expression


<IPython.core.display.Latex object>

### Unique labels

In [19]:
_d = {
    J: "moment of inertia",
    E: "modulus of elasticity",
    a: "an expression",
}

# generate the labels from the description
_l = generate_label(_d, unique_id=True)
print(_l)

# pass _l to show_eqn
show_eqn(_p, label=_l)


{J: 'eq-534vb10v', E: 'eq-1sipf20t', a: 'eq-30phqdq8'}
J: eq-534vb10v
E: eq-1sipf20t
a: eq-30phqdq8


<IPython.core.display.Latex object>

## Full automatic generation

In [20]:
# pass _l to show_eqn
show_eqn([_p, _d], label=generate_unique_label)


J: eq-3lb6w5uv
E: eq-1mp6ez2n
a: eq-2mla7kuo


<IPython.core.display.Latex object>